# Test 2 — International results only (`data/international/results`)

Feature engineering for **match-level** modeling using **only** the international results file (same columns as `results.csv`: date, teams, scores, tournament, city, country, neutral).

**Databricks:** keep `%python` as the first line of each code cell. Locally, remove `%python`.

**Load source (pick one):**
- **`USE_CSV = True`:** upload `results.csv` to DBFS / a Volume and set `INTERNATIONAL_CSV`.
- **`USE_CSV = False`:** read from your federated BigQuery table `analytics.international_results` (after the R upload script).

**Features (no StatsBomb / no xG in this file):** goals for/against per team-match, **rolling 5-match** avg goals for/against and win rate, **neutral** flag, **H2H** prior meeting count, **outcome** label when scores exist.

**Saved Delta tables:** `intl_rq4_team_long`, `intl_rq4_match_modeling`.

In [0]:
%python
from pyspark.sql import functions as F
from pyspark.sql.window import Window

BQ_CATALOG = "bq_raw_statsbomb_sa_catalog"  # UC catalog that can see BigQuery `analytics`
INTERNATIONAL_TABLE = f"{BQ_CATALOG}.analytics.international_results"

OUTPUT_CATALOG = "main"
OUTPUT_SCHEMA = "football_rq4"
OUT = f"{OUTPUT_CATALOG}.{OUTPUT_SCHEMA}"

ROLL_N = 5

## 1) Load international results

In [0]:
%python
raw = spark.table(INTERNATIONAL_TABLE)

intl = (
    raw.select(
        F.trim(F.col("date").cast("string")).alias("date_s"),
        F.trim(F.col("home_team").cast("string")).alias("home_team"),
        F.trim(F.col("away_team").cast("string")).alias("away_team"),
        F.col("home_score").cast("int").alias("home_score"),
        F.col("away_score").cast("int").alias("away_score"),
        F.trim(F.col("tournament").cast("string")).alias("tournament"),
        F.trim(F.col("city").cast("string")).alias("city"),
        F.trim(F.col("country").cast("string")).alias("country"),
        F.lower(F.trim(F.col("neutral").cast("string"))).alias("neutral_s"),
    )
    .withColumn("match_date", F.to_date(F.col("date_s"), "yyyy-MM-dd"))
    .withColumn(
        "neutral",
        F.when(F.col("neutral_s").isin("true", "t", "1", "yes"), True)
        .when(F.col("neutral_s").isin("false", "f", "0", "no"), False)
        .otherwise(F.lit(None).cast("boolean")),
    )
    .drop("date_s", "neutral_s")
    .filter(F.col("home_team") != F.lit(""))
    .filter(F.col("away_team") != F.lit(""))
)

intl = intl.withColumn(
    "match_id",
    F.xxhash64(
        F.concat_ws(
            "|",
            F.col("match_date").cast("string"),
            F.col("home_team"),
            F.col("away_team"),
        )
    ),
)

intl.createOrReplaceTempView("intl_matches")
n = intl.count()
n_done = intl.filter(
    F.col("home_score").isNotNull() & F.col("away_score").isNotNull()
).count()
print(n, "rows total;", n_done, "with both scores (completed);", n - n_done, "scheduled / unknown scores")
print("Sample of completed matches (scores non-null):")
display(
    intl.filter(F.col("home_score").isNotNull() & F.col("away_score").isNotNull())
    .orderBy(F.asc("match_date"))
    .limit(15)
)
print("Earliest rows in the table (chronological start — scores present):")
display(intl.orderBy(F.asc("match_date")).limit(8))

## 2) Long format: one row per (match, team) with goals for / against and win flag

In [0]:
%python
home_side = intl.select(
    "match_id",
    "match_date",
    "tournament",
    "city",
    "country",
    "neutral",
    F.col("home_team").alias("team"),
    F.col("away_team").alias("opponent"),
    F.lit(True).alias("is_home"),
    "home_score",
    "away_score",
    F.col("home_score").alias("goals_for"),
    F.col("away_score").alias("goals_against"),
    F.when(
        F.col("home_score").isNotNull() & F.col("away_score").isNotNull(),
        F.when(F.col("home_score") > F.col("away_score"), 1).otherwise(0),
    ).alias("win"),
)

away_side = intl.select(
    "match_id",
    "match_date",
    "tournament",
    "city",
    "country",
    "neutral",
    F.col("away_team").alias("team"),
    F.col("home_team").alias("opponent"),
    F.lit(False).alias("is_home"),
    "home_score",
    "away_score",
    F.col("away_score").alias("goals_for"),
    F.col("home_score").alias("goals_against"),
    F.when(
        F.col("home_score").isNotNull() & F.col("away_score").isNotNull(),
        F.when(F.col("away_score") > F.col("home_score"), 1).otherwise(0),
    ).alias("win"),
)

team_long = home_side.unionByName(away_side)
team_long.createOrReplaceTempView("intl_team_long")
display(team_long.orderBy(F.asc("match_date")).limit(12))

## 3) Rolling 5 prior matches per team (no leakage: excludes current match)

In [0]:
%python
w = (
    Window.partitionBy("team")
    .orderBy(F.col("match_date").asc_nulls_last(), F.col("match_id").asc())
    .rowsBetween(-ROLL_N, -1)
)

team_roll = (
    team_long.withColumn("roll_gf", F.avg("goals_for").over(w))
    .withColumn("roll_ga", F.avg("goals_against").over(w))
    .withColumn("roll_win_rate", F.avg(F.col("win").cast("double")).over(w))
)

team_roll.createOrReplaceTempView("intl_team_roll")
display(team_roll.filter(F.col("roll_gf").isNotNull()).orderBy(F.asc("match_date")).limit(12))

## 4) H2H prior count + match-level modeling frame (one row per fixture)

In [0]:
%python
base = intl.select(
    F.col("match_id").alias("mid"),
    "match_date",
    F.col("home_team").alias("h"),
    F.col("away_team").alias("a"),
)

h2h = (
    base.alias("m1")
    .join(
        base.alias("m2"),
        (
            (
                ((F.col("m2.h") == F.col("m1.h")) & (F.col("m2.a") == F.col("m1.a")))
                | ((F.col("m2.h") == F.col("m1.a")) & (F.col("m2.a") == F.col("m1.h")))
            )
            & (F.col("m2.match_date") < F.col("m1.match_date"))
        ),
        how="left",
    )
    .groupBy("m1.mid")
    .agg(F.countDistinct("m2.mid").alias("h2h_prior_meetings"))
    .select(F.col("mid").alias("match_id"), "h2h_prior_meetings")
)

rh = (
    team_roll.filter(F.col("is_home"))
    .select(
        "match_id",
        F.col("roll_gf").alias("home_roll_gf"),
        F.col("roll_ga").alias("home_roll_ga"),
        F.col("roll_win_rate").alias("home_roll_win_rate"),
    )
    .dropDuplicates(["match_id"])
)

ra = (
    team_roll.filter(~F.col("is_home"))
    .select(
        "match_id",
        F.col("roll_gf").alias("away_roll_gf"),
        F.col("roll_ga").alias("away_roll_ga"),
        F.col("roll_win_rate").alias("away_roll_win_rate"),
    )
    .dropDuplicates(["match_id"])
)

fm = intl.select(
    "match_id",
    "match_date",
    "home_team",
    "away_team",
    "home_score",
    "away_score",
    "tournament",
    "city",
    "country",
    "neutral",
)

match_modeling = (
    fm.join(rh, on="match_id", how="left")
    .join(ra, on="match_id", how="left")
    .join(h2h, on="match_id", how="left")
    .withColumn(
        "outcome_label",
        F.when(
            F.col("home_score").isNull() | F.col("away_score").isNull(),
            F.lit(None).cast("string"),
        )
        .when(F.col("home_score") > F.col("away_score"), F.lit("home_win"))
        .when(F.col("home_score") < F.col("away_score"), F.lit("away_win"))
        .otherwise(F.lit("draw")),
    )
)

match_modeling.createOrReplaceTempView("intl_rq4_match_modeling")
display(match_modeling.filter(F.col("outcome_label").isNotNull()).orderBy(F.asc("match_date")).limit(15))

## 5) Save Delta tables

In [0]:
%python
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {OUTPUT_CATALOG}.{OUTPUT_SCHEMA}")

team_long.write.mode("overwrite").format("delta").saveAsTable(
    f"{OUT}.intl_rq4_team_long"
)
team_roll.write.mode("overwrite").format("delta").saveAsTable(
    f"{OUT}.intl_rq4_team_roll"
)
match_modeling.write.mode("overwrite").format("delta").saveAsTable(
    f"{OUT}.intl_rq4_match_modeling"
)

print("Saved:")
print(f"  {OUT}.intl_rq4_team_long")
print(f"  {OUT}.intl_rq4_team_roll")
print(f"  {OUT}.intl_rq4_match_modeling")

## 6) Row counts

In [0]:
%python
for name in ("intl_team_long", "intl_team_roll", "intl_rq4_match_modeling"):
    print(name, spark.table(name).count())